In [1]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image
from pathlib import Path
import torch
from torch.utils.data import Dataset

e:\Github Projects (From August 2026)\legal-idp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.append("../src")  # adjust relative path based on where your notebook lives

from data_utils import ANN_DIR, IMG_DIR, extract_qa_pairs, pairs_to_donut_target

In [11]:
MODEL_NAME = "naver-clova-ix/donut-base"

processor = DonutProcessor.from_pretrained(MODEL_NAME)
# Set the image size to 720x960 (width x height) for the processor because the original Donut model was trained on images of this size 
# which is too large for my 6GB VRAM GPU. The model will still work with smaller images, but the performance may be affected.
processor.image_processor.size = {"height": 960, "width": 720}

model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 484/484 [00:00<00:00, 15534.93it/s]


In [12]:
# Register our custom tags as special tokens - without this, the tokenizer
# would split "<s_question>" into several sub-word pieces instead of treating
# it as one atomic structural marker
new_tokens = ["<s_question>", "</s_question>", "<s_answer>", "</s_answer>"]
processor.tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})

4

In [13]:
# model's embedding matrix must be resized to match the tokenizer's new vocab size,
# since we just added 4 tokens it didn't have before
model.decoder.resize_token_embeddings(len(processor.tokenizer))

MBartScaledWordEmbedding(57529, 1024, padding_idx=1)

In [14]:
def build_donut_examples():
    """
    Build (stem, target_string) pairs for every FUNSD doc that has at least
    one valid question->answer link. Docs with zero pairs are skipped -
    nothing meaningful for Donut to learn to generate from them.
    """
    examples = []
    for ann_file in ANN_DIR.glob("*.json"):
        stem = ann_file.stem
        pairs = extract_qa_pairs(stem)
        if not pairs:
            continue
        target = pairs_to_donut_target(pairs)
        examples.append((stem, target))
    return examples

In [15]:
class FUNSDDonutDataset(Dataset):
    """
    Wraps FUNSD documents for Donut fine-tuning: image in, tagged
    question/answer string out. No boxes involved - Donut reads the
    raw image directly.
    """

    def __init__(self, examples, max_length=512):
        self.examples = examples
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        stem, target = self.examples[idx]

        img_path = next(IMG_DIR.glob(f"{stem}.*"))
        image = Image.open(img_path).convert("RGB")

        # image -> pixel_values, same processor handles Donut's vision encoder input
        pixel_values = processor(image, return_tensors="pt").pixel_values.squeeze(0)

        # text target -> token ids. add_special_tokens=True adds the model's
        # own BOS/EOS (not our custom tags, those are already literal tokens now)
        labels = processor.tokenizer(
            target,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)

        # padding tokens must be -100 so they're ignored in the loss,
        # same convention as LayoutLMv3's -100 labels
        labels[labels == processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}

In [16]:
examples = build_donut_examples()
print(f"Docs with at least one Q-A pair: {len(examples)} / 149")

Docs with at least one Q-A pair: 147 / 149


In [17]:
dataset = FUNSDDonutDataset(examples)
sample = dataset[0]
print("pixel_values:", sample["pixel_values"].shape)
print("labels:", sample["labels"].shape)

pixel_values: torch.Size([3, 960, 720])
labels: torch.Size([512])


In [18]:
print(processor.image_processor.size)

{'height': 960, 'width': 720}


In [19]:
import random

def get_donut_train_val_split(examples, val_ratio=0.15, seed=42):
    """
    Split (stem, target) examples into train/val subsets.
    Same fixed-seed approach as LayoutLMv3's split, for consistency
    across both pipelines - though note this is a DIFFERENT split
    (147 docs here vs 149 for LayoutLMv3), since 2 docs had no Q-A pairs.
    """
    examples = list(examples)  # copy, avoid mutating the original in place
    rng = random.Random(seed)
    rng.shuffle(examples)

    val_size = int(len(examples) * val_ratio)
    val_examples = examples[:val_size]
    train_examples = examples[val_size:]

    return train_examples, val_examples

In [20]:
examples = build_donut_examples()
train_examples, val_examples = get_donut_train_val_split(examples)
print(f"Train: {len(train_examples)}, Val: {len(val_examples)}")

Train: 125, Val: 22


In [21]:
train_dataset = FUNSDDonutDataset(train_examples)
val_dataset = FUNSDDonutDataset(val_examples)

In [22]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./checkpoints/donut-funsd",
    per_device_train_batch_size=1,   # tight VRAM budget, high-res images
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size = 8
    num_train_epochs=10,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,       # enables autoregressive generation at eval time
    fp16=True,
    logging_steps=10,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    # compute_metrics left out for now - we'll add it once training runs cleanly
)

In [24]:
# Set model config attributes to match the processor's tokenizer, so that
# generation works correctly. This is a bit of a hack, but it works.

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id if processor.tokenizer.bos_token_id is not None else processor.tokenizer.cls_token_id
model.config.vocab_size = model.decoder.config.vocab_size

In [ ]:
trainer.train()